# **US Flights Delay:** Schema Engineering

## Imports

In [1]:
import sys

In [5]:
sys.path.append("../src/database")
sys.path.append("../src/database/queries")

In [6]:
import connection 
import base_queries

## Connect to client and database

In [9]:
MONGODB_URI = "mongodb://localhost:27017/"
MONGODB_NAME = "flights_delay_db"

In [10]:
client, database = connection.connect_to_database(uri = MONGODB_URI, db_name = MONGODB_NAME)

In [11]:
database.list_collection_names()

['airports',
 'airports_summary_hybrid_optimized',
 'runways',
 'cancelled_deverted_2023',
 'flights_hybrid_optimized',
 'us_flights_2023',
 'weather_hybrid_optimized',
 'airport_frequencies',
 'airports_geolocation',
 'us_flights_optimized',
 'weather_meteo_by_airport']

## Queries

**Upit 1:** Koje američke države imaju najveća prosečna kašnjenja po sezonama (Winter, Spring, Summer, Fall)?  
  - Analizira se sezonalnost kašnjenja — da li se kašnjenja razlikuju po godišnjim dobima.  
  - Koriste se kolekcije `us_flights_2023` i `airports_geolocation`.  
  - Faze obrade:
    1. Odabir letova sa validnim datumom, aerodromom i kašnjenjem.  
    2. Iz datuma se izvlači mesec i dodeljuje sezona (`Winter`, `Spring`, `Summer`, `Fall`).  
    3. `$lookup` sa `airports_geolocation` radi određivanja države (`state`).  
    4. Grupisanje po državi i sezoni radi računanja prosečnog kašnjenja.  
  - **Značaj:** omogućava uvid u sezonske obrasce kašnjenja i identifikuje države koje su najviše pogođene vremenskim faktorima tokom određenih perioda godine.

---

**Upit 2:** Koje avio-kompanije imaju najveća prosečna kašnjenja tokom dana sa padavinama?  
  - Cilj je proceniti uticaj vremenskih uslova na tačnost letova po avio-kompanijama.  
  - Koriste se kolekcije `us_flights_2023` i `weather_meteo_by_airport`.  
  - **Značajne padavine:** definišu se kao `prcp > 5 mm`, što meteorološki označava **umerene do jake padavine**.  
  - Faze obrade:
    1. Filtriraju se letovi sa poznatom avio-kompanijom i aerodromom.  
    2. `$lookup` sa vremenskim uslovima na dan leta (`weather_meteo_by_airport`).  
    3. Filtriranje letova gde su padavine > 5 mm.  
    4. Grupisanje po avio-kompaniji radi izračunavanja prosečnog kašnjenja i broja letova u tim uslovima.  
  - **Značaj:** pokazuje koje avio-kompanije su najosetljivije na vremenske uslove i gde postoji prostor za operativno poboljšanje.


---

**Upit 3:** Koji aerodromi imaju najviše otkazanih letova tokom loših vremenskih uslova?  
  - Povezuje informacije o otkazanim letovima i meteorološkim uslovima.  
  - Koriste se kolekcije `us_flights_2023`, `weather_meteo_by_airport` i `airports_geolocation`.  
  - **Loše vreme:** definisano kao:
    - `prcp > 10 mm` → jake ili vrlo jake padavine  
    - `wspd > 15 m/s` → jak do olujni vetar  
  - Faze obrade:
    1. Filtriranje letova sa `cancelled = 1`.  
    2. `$lookup` sa `weather_meteo_by_airport` radi vremenskih uslova na aerodromu.  
    3. Zadržavanje samo dana sa lošim vremenom (padavine ili vetar).  
    4. `$lookup` sa `airports_geolocation` radi određivanja grada i države.  
    5. Grupisanje po aerodromu i brojanje otkazanih letova.  
  - **Značaj:** identifikuje aerodrome koji su najugroženiji ekstremnim vremenskim prilikama i gde su potrebne bolje mere za smanjenje rizika.

---

- **Upit 4:** Da li aerodromi sa raznovrsnijim pistama imaju manja prosečna kašnjenja?  
  - Analizira vezu između infrastrukture aerodroma i efikasnosti letova.  
  - Koriste se kolekcije `airports`, `runways` i `us_flights_2023`.  
  - Uvodi se **Runway Diversity Index (RDI)**:
    - predstavlja broj različitih površina pista (`surface`) po aerodromu.  
  - Faze obrade:
    1. `$lookup` između `airports` i `runways` radi dobijanja svih pista.  
    2. Grupisanje po aerodromu i računanje broja jedinstvenih površina (`RDI`).  
    3. Izračunavanje prosečne dužine pista.  
    4. `$lookup` sa `us_flights_2023` radi prosečnog kašnjenja po aerodromu.  
  - **Značaj:** omogućava uvid u to da li raznovrsnost pista doprinosi boljoj operativnoj fleksibilnosti i smanjenju kašnjenja.
  
---

**Upit 5:** Kako se performanse avio-kompanija razlikuju po tipu udaljenosti leta (Short, Medium, Long Haul)?  
  - Poredi prosečna kašnjenja po tipu rute (`distance_type`).  
  - Koristi se kolekcija `us_flights_2023`.  
  - Faze obrade:
    1. Filtriranje letova sa definisanim `distance_type`, `airline` i `dep_delay`.  
    2. Grupisanje po `airline` i `distance_type`.  
    3. Računanje prosečnog kašnjenja i broja letova.  
    4. Sortiranje po prosečnom kašnjenju za lakšu analizu.  
  - **Značaj:** omogućava poređenje performansi avio-kompanija u zavisnosti od dužine rute, i otkriva gde se javljaju veći operativni problemi.


In [ ]:
base_queries = base_queries.get_base_queries()

In [ ]:
base_query_1 = base_queries['query_1']

In [ ]:
base_query_2 = base_queries['query_2']

In [ ]:
base_query_3 = base_queries['query_3']

In [ ]:
base_query_4 = base_queries['query_4']

In [ ]:
base_query_5 = base_queries['query_5']